In [ ]:
!pip install whisperx
!pip install torch torchvision torchaudio

In [ ]:
import os
import gc
import re
import html
import torch
import email.utils
import urllib.request
import xml.etree.ElementTree as ET
from google.colab import userdata
import whisperx
from whisperx.diarize import DiarizationPipeline

# =====================================================================
# 📋 SYSTEM CONFIGURATION & PATTERN SCHEMAS
# =====================================================================
XML_FILE_PATH = "feed.xml"  # Your uploaded 2.7MB RSS data map
HF_TOKEN = userdata.get('HF_TOKEN')

# Batch cadence — write a zip to Google Drive every N successful transcripts so 20–40
# episode runs don't fill Colab disk or risk losing work if the runtime is recycled.
BATCH_SIZE = 5

# Staging folder for batch zips on the Colab runtime disk. Created if missing.
# Drive is blocked at the org level and files.download() hangs on long runs, so zips
# accumulate here for manual download via the Colab sidebar (Files panel → right-click → Download).
OUTPUT_DIR = "/content/batches"

# Fine-tuned Industry Acronym Bias (Passed during model load).
# Bias list covers industry acronyms + recurring hosts/guests/orgs that Whisper systematically mishears.
# Stay under Whisper's ~224-token initial_prompt budget — add high-recurrence names first.
# condition_on_previous_text=False is the primary anti-loop mitigation: stops each segment from being
# primed by the model's own (possibly looping) prior output. Slight long-context coherence tradeoff;
# imperceptible on episodic podcast content with clear turn-changes.
ASR_OPTIONS = {
    "initial_prompt": (
        "HCAHPS, TPS Report, Touchpoint Podcast, Press Ganey Forsta, Qualtrics, CMO, digital front door, "
        "Chris Boyer, Reed Smith, MyChart, "
        "Carrie Liken, Steve Koch, Cast and Hue, Martha Van Berkel, Schema App, John Berndt, Valtech, "
        "Bryan Vartabedian, Amsive Health, Craig Blake, Jane Crosby, True North Custom, "
        "Brandon Scott, TenAdams, Ray Mina, Freshpaint, Dean Browell, Feedback, Danny Fell, Unlock Health, "
        "Snowflake, Yext"
    ),
    "condition_on_previous_text": False,
}

# Voice Activity Detection constraints to stop truncation/drops
VAD_OPTIONS = {
    "vad_onset": 0.500,
    "vad_offset": 0.363
}

# Runtime Speaker Heuristic Identification Pattern Layouts
ANNOUNCER_PATTERNS = [
    r"\bwelcome to touchpoint\b",
    r"\ba podcast dedicated to discussions on digital marketing\b",
]
CHRIS_SELF_PATTERNS = [r"\b(I'm Chris( Boyer)?|Chris Boyer here|Chris here|this is Chris)\b"]
# Two pattern shapes for addressing the cohost vocatively:
# (a) leading word + name ("welcome Chris", "thanks Chris", etc.)
# (b) vocative at sentence/clause boundary ("Chris, ..." at start of turn or after ". ")
CHRIS_ADDRESSED_PATTERNS = [
    # (a) leading address word + name, e.g. "thanks Chris" / "so Chris"
    r"\b(?:welcome|thanks|right|hey|so|but|well|yeah|okay|ok|alright|now|and)\s+Chris(?=[\s,.!?]|$)",
    # (b) vocative at sentence/clause boundary REQUIRING trailing punctuation to disambiguate
    # from third-person subject ("Chris and I went..." should NOT match; "Chris, pour..." SHOULD)
    r"(?:^|[.!?]\s+|,\s+)Chris(?=[,.!?])",
]
REED_SELF_PATTERNS = [r"\b(I'm Reed( Smith)?|Reed Smith here|Reed here|this is Reed)\b"]
REED_ADDRESSED_PATTERNS = [
    r"\b(?:welcome|thanks|right|hey|so|but|well|yeah|okay|ok|alright|now|and)\s+Reed(?=[\s,.!?]|$)",
    r"(?:^|[.!?]\s+|,\s+)Reed(?=[,.!?])",
]
# =====================================================================

def slugify(text):
    """Convert text to lowercase kebab-case for URL slug construction."""
    text = text.lower()
    text = re.sub(r"['\"]", "", text)
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-")

def parse_content_encoded(html_text):
    """Parses content:encoded HTML into prose description + a simple resources list.
    Each resource is emitted as {url, raw_text} — downstream consumers do their own
    structured parsing. Touch Point's show-notes grammar (publisher/title/publication/date)
    varies enough across episodes that regex-splitting in Python is unreliable; raw text
    plus URL gives Phase 1 (Claude) everything it needs to compose Episode Notes."""
    if not html_text:
        return "  No description provided.", []

    resources = []
    li_matches = re.findall(r'<li[^>]*>(.*?)</li>', html_text, re.DOTALL | re.IGNORECASE)
    for li in li_matches:
        a_match = re.search(r'<a\s+[^>]*href=["\']([^"\']+)["\']', li, re.DOTALL | re.IGNORECASE)
        if not a_match:
            continue
        url = a_match.group(1)

        # Strip the entire <a>...</a> block (including its link text, which is just the URL repeated)
        # and any remaining HTML tags to get the prose surrounding the link.
        text_only = re.sub(r'<a\s+[^>]*>.*?</a>', '', li, flags=re.DOTALL | re.IGNORECASE)
        text_only = re.sub(r'<[^>]+>', '', text_only)
        text_only = html.unescape(text_only).strip()
        # Trim trailing artifacts (the prose typically ends with a colon before the URL)
        text_only = text_only.rstrip(':').rstrip('-').rstrip('–').rstrip(',').strip()

        resources.append({
            "url": url,
            "raw_text": text_only if text_only else "(no description)"
        })
            
    text_processed = re.sub(r'<br\s*/?>', '\n', html_text, flags=re.IGNORECASE)
    text_processed = re.sub(r'</p\s*>', '\n\n', text_processed, flags=re.IGNORECASE)
    text_processed = re.sub(r'<li[^>]*>', '• ', text_processed, flags=re.IGNORECASE)
    text_processed = re.sub(r'</li>', '\n', text_processed, flags=re.IGNORECASE)
    text_processed = re.sub(r'<[^>]+>', '', text_processed)  
    
    desc_lines = []
    for line in text_processed.splitlines():
        line_clean = html.unescape(line).strip()
        if line_clean:
            desc_lines.append(f"  {line_clean}")
            
    description_block = "\n".join(desc_lines) if desc_lines else "  No description provided."
    return description_block, resources

def log_ambiguity_warning(reason, announcer_tag, chris_votes, reed_votes):
    """Logs a non-fatal warning to the terminal when speaker resolution is muddy."""
    print("\n⚠️ " + "=*="*20)
    print(f"  HEURISTIC AMBIGUITY NOTICE: {reason}")
    print(f"  Announcer Tag: {announcer_tag} | Chris Votes: {chris_votes} | Reed Votes: {reed_votes}")
    print("  System will preserve raw SPEAKER_NN tags for unverified voices.")
    print("=*="*20 + "\n")

def resolve_speaker_identities(segments):
    """Heuristically maps voices. Falls back to original SPEAKER_NN tags on conflicts."""
    all_tags = set(seg.get("speaker") for seg in segments if seg.get("speaker"))
    
    # Initialize the map with the raw tags themselves as the safe fallback baseline
    speaker_map = {tag: tag for tag in all_tags}
    
    # 1. Resolve Announcer
    announcer_votes = {spk: 0 for spk in all_tags}
    for seg in segments:
        text = seg.get("text", "")
        speaker_id = seg.get("speaker")
        if not speaker_id:
            continue
        if re.search(r"\ba podcast dedicated to discussions on digital marketing\b", text, re.IGNORECASE):
            announcer_votes[speaker_id] += 10  
        elif re.search(r"\bwelcome to touchpoint\b", text, re.IGNORECASE):
            announcer_votes[speaker_id] += 1   

    max_announcer_votes = max(announcer_votes.values()) if announcer_votes else 0
    announcer_tag = None

    if max_announcer_votes > 0:
        announcer_candidates = [spk for spk, v in announcer_votes.items() if v == max_announcer_votes]
        if len(announcer_candidates) == 1:
            candidate_tag = announcer_candidates[0]
            # Sanity-check: a real Announcer cluster contains only the boilerplate intro/outro and
            # ad reads — a handful of short turns. If the boilerplate-winning cluster also contains
            # many substantive (>30 word, non-boilerplate) turns, PyAnnote merged a host or guest
            # into the Announcer cluster. Observed across 6 episodes in the May 2025 → May 2026
            # catch-up batch (TP467, TP470, TP475, TP480, TP481, TP484). In that case, exclude the
            # cluster from BOTH announcer attribution AND host voting — leave the raw SPEAKER_NN
            # tag so Phase 1 surfaces it for manual review rather than confidently mislabeling.
            boilerplate_re = re.compile(
                r"\bwelcome to touchpoint\b|\ba podcast dedicated to discussions on digital marketing\b",
                re.IGNORECASE,
            )
            substantive_count = 0
            for seg in segments:
                if seg.get("speaker") != candidate_tag:
                    continue
                seg_text = seg.get("text", "")
                if boilerplate_re.search(seg_text):
                    continue
                if len(seg_text.split()) > 30:
                    substantive_count += 1
            if substantive_count > 5:
                print(
                    f"\n  ⚠️ ANNOUNCER CLUSTER COLLISION: {candidate_tag} has {substantive_count} "
                    f"substantive non-boilerplate turns. PyAnnote likely merged a host or guest "
                    f"into the Announcer cluster. Excluding cluster from attribution; raw label "
                    f"preserved for Phase 1 manual review."
                )
                announcer_tag = candidate_tag  # excludes cluster from host voting below
                # speaker_map[candidate_tag] stays as the raw SPEAKER_NN tag (default init)
            else:
                announcer_tag = candidate_tag
                speaker_map[announcer_tag] = "Announcer"
        else:
            log_ambiguity_warning("Splitting announcer signatures detected.", announcer_candidates, {}, {})

    non_announcer_tags = [t for t in all_tags if t != announcer_tag]
    if not non_announcer_tags:
        return speaker_map

    # 2. Tally Host Context Points (self-intros + vocative addressing)
    chris_votes = {spk: 0 for spk in non_announcer_tags}
    reed_votes = {spk: 0 for spk in non_announcer_tags}

    for seg in segments:
        current_speaker = seg.get("speaker")
        if not current_speaker or current_speaker == announcer_tag:
            continue
        text = seg.get("text", "")

        # Self-intros score the CURRENT speaker
        if any(re.search(pat, text, re.IGNORECASE) for pat in CHRIS_SELF_PATTERNS):
            chris_votes[current_speaker] += 3
        if any(re.search(pat, text, re.IGNORECASE) for pat in REED_SELF_PATTERNS):
            reed_votes[current_speaker] += 3

        # Vocative addressing scores the OTHER non-announcer speaker
        other_hosts = [s for s in non_announcer_tags if s != current_speaker]
        for other_spk in other_hosts:
            if any(re.search(pat, text, re.IGNORECASE) for pat in CHRIS_ADDRESSED_PATTERNS):
                chris_votes[other_spk] += 2
            if any(re.search(pat, text, re.IGNORECASE) for pat in REED_ADDRESSED_PATTERNS):
                reed_votes[other_spk] += 2

    # 3. Apply assignment logic with safe fallbacks
    if len(non_announcer_tags) == 1:
        # Solo episode — only self-intros and third-person mentions score
        solo_spk = non_announcer_tags[0]
        c_score = chris_votes[solo_spk]
        r_score = reed_votes[solo_spk]

        # Bonus: "Reed and I" / "Chris and I" — strong signal the speaker is the OTHER host
        for seg in segments:
            if seg.get("speaker") != solo_spk:
                continue
            text = seg.get("text", "")
            if re.search(r"\bReed and I\b", text, re.IGNORECASE):
                c_score += 1
            if re.search(r"\bChris and I\b", text, re.IGNORECASE):
                r_score += 1

        if c_score != r_score:
            speaker_map[solo_spk] = "Chris Boyer" if c_score > r_score else "Reed Smith"
        else:
            log_ambiguity_warning("Solo host identity tie/zero score. Leaving as raw tag.",
                                  announcer_tag, chris_votes, reed_votes)
    else:
        # Two-host (or more) case
        sum_c = sum(chris_votes.values())
        sum_r = sum(reed_votes.values())

        if sum_c > 0 and sum_r > 0:
            best_chris = max(chris_votes, key=chris_votes.get)
            best_reed = max(reed_votes, key=reed_votes.get)

            if best_chris != best_reed:
                speaker_map[best_chris] = "Chris Boyer"
                speaker_map[best_reed] = "Reed Smith"
            elif len(non_announcer_tags) == 2:
                # Same tag won both host profiles — disambiguate by signal dominance on the
                # contested tag, then assign the other tag by exclusion. (TP488 had this:
                # SPEAKER_02 won both, but Reed=17 vs Chris=3 on it — Reed clearly dominates.)
                contested = best_chris
                other_tag = [t for t in non_announcer_tags if t != contested][0]
                c_on_contested = chris_votes[contested]
                r_on_contested = reed_votes[contested]
                if r_on_contested >= 2 * c_on_contested and r_on_contested >= 3:
                    speaker_map[contested] = "Reed Smith"
                    speaker_map[other_tag] = "Chris Boyer"
                    print(f"  ℹ️ Disambiguated by signal dominance on {contested}: Reed={r_on_contested} vs Chris={c_on_contested}. {other_tag} → Chris by exclusion.")
                elif c_on_contested >= 2 * r_on_contested and c_on_contested >= 3:
                    speaker_map[contested] = "Chris Boyer"
                    speaker_map[other_tag] = "Reed Smith"
                    print(f"  ℹ️ Disambiguated by signal dominance on {contested}: Chris={c_on_contested} vs Reed={r_on_contested}. {other_tag} → Reed by exclusion.")
                else:
                    log_ambiguity_warning("Identity overlap with comparable signal strength on contested tag. Falling back to raw labels.",
                                          announcer_tag, chris_votes, reed_votes)
            else:
                log_ambiguity_warning("Identity overlap: same tag won both host profiles (3+ speakers). Falling back to raw labels.",
                                      announcer_tag, chris_votes, reed_votes)
        elif sum_c > 0 and sum_r == 0 and len(non_announcer_tags) == 2:
            # Chris identified by contextual markers; Reed by exclusion.
            # Touch Point is a 2-host show — encode that domain assumption.
            best_chris = max(chris_votes, key=chris_votes.get)
            other_tag = [t for t in non_announcer_tags if t != best_chris][0]
            speaker_map[best_chris] = "Chris Boyer"
            speaker_map[other_tag] = "Reed Smith"
            print(f"  ℹ️ Reed identified by exclusion ({other_tag}); Chris had {chris_votes[best_chris]} votes.")
        elif sum_r > 0 and sum_c == 0 and len(non_announcer_tags) == 2:
            # Reed identified; Chris by exclusion.
            best_reed = max(reed_votes, key=reed_votes.get)
            other_tag = [t for t in non_announcer_tags if t != best_reed][0]
            speaker_map[best_reed] = "Reed Smith"
            speaker_map[other_tag] = "Chris Boyer"
            print(f"  ℹ️ Chris identified by exclusion ({other_tag}); Reed had {reed_votes[best_reed]} votes.")
        else:
            log_ambiguity_warning("Neither host returned contextual markers across transcript timeline. Falling back to raw labels.",
                                  announcer_tag, chris_votes, reed_votes)

    return speaker_map

def parse_rss_metadata(xml_path):
    """Parses the RSS XML feed to map MP3 file links to highly rich structural metadata."""
    metadata_map = {}
    if not os.path.exists(xml_path):
        print(f"❌ Critical Error: '{xml_path}' not found. Please upload your RSS feed XML file to the side panel.")
        return metadata_map

    print(f"Parsing {xml_path} for structural identity mappings...")
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        itunes_ns = "{http://www.itunes.com/dtds/podcast-1.0.dtd}"
        content_ns = "{http://purl.org/rss/1.0/modules/content/}encoded"
        
        summary_tag = f"{itunes_ns}summary"
        duration_tag = f"{itunes_ns}duration"

        for item in root.findall(".//item"):
            title_elem = item.find("title")
            title = title_elem.text if title_elem is not None else "Unknown Title"
            link_elem = item.find("link")
            link = link_elem.text if link_elem is not None else ""
            guid_elem = item.find("guid")
            guid_text = guid_elem.text if guid_elem is not None else ""

            pub_date_elem = item.find("pubDate")
            pub_date_iso = "null"
            if pub_date_elem is not None and pub_date_elem.text:
                try:
                    parsed_dt = email.utils.parsedate_to_datetime(pub_date_elem.text)
                    pub_date_iso = parsed_dt.strftime("%Y-%m-%d")
                except Exception:
                    pub_date_iso = "null"

            dur_elem = item.find(duration_tag)
            duration_text = dur_elem.text.strip() if dur_elem is not None and dur_elem.text else ""
            duration_seconds = "null"
            duration_hhmmss = "null"
            
            if duration_text:
                if duration_text.isdigit():
                    total_secs = int(duration_text)
                    duration_seconds = total_secs
                    h = total_secs // 3600
                    m = (total_secs % 3600) // 60
                    s = total_secs % 60
                    duration_hhmmss = f'"{h:02d}:{m:02d}:{s:02d}"'
                elif ":" in duration_text:
                    parts = duration_text.split(":")
                    if len(parts) == 2:
                        m, s = int(parts[0]), int(parts[1])
                        total_secs = m * 60 + s
                        h = 0
                    elif len(parts) == 3:
                        h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
                        total_secs = h * 3600 + m * 60 + s
                    duration_seconds = total_secs
                    duration_hhmmss = f'"{h:02d}:{m:02d}:{s:02d}"'

            content_elem = item.find(content_ns)
            desc_elem = item.find("description")
            sum_elem = item.find(summary_tag)
            
            html_payload = ""
            if content_elem is not None and content_elem.text:
                html_payload = content_elem.text
            elif desc_elem is not None and desc_elem.text:
                html_payload = desc_elem.text
            elif sum_elem is not None and sum_elem.text:
                html_payload = sum_elem.text
                
            indented_desc, resources_list = parse_content_encoded(html_payload)

            enclosure = item.find("enclosure")
            url_filename = ""
            audio_url = ""
            if enclosure is not None and "url" in enclosure.attrib:
                audio_url = enclosure.attrib["url"]
                url_filename = audio_url.split("/")[-1].split("?")[0]

            if not url_filename:
                continue

            ep_match = re.search(r'\b(TP\d+)\b', title, re.IGNORECASE)
            if not ep_match and guid_text:
                ep_match = re.search(r'\b(TP\d+)\b', guid_text, re.IGNORECASE)
            if not ep_match:
                ep_match = re.search(r'\b(tp\d+)\b', url_filename, re.IGNORECASE)
            
            if ep_match:
                ep_num = ep_match.group(1).upper()
            else:
                ep_num = "TPXXX"

            is_icymi = bool(re.search(r'\bICYMI\b', title, re.IGNORECASE))
            icymi_original = "null"
            if is_icymi and html_payload:
                # Original episode is referenced in the description, not the title.
                # Try TP### first, then "episode N" / "ep N" / "#N". Skip matches that equal
                # the current ep_num (the current episode's own TP### appears in the title).
                current_num = ep_num.replace("TP", "")
                for pattern in [r'\bTP(\d{2,4})\b', r'\b(?:episode|ep|#)\s*(\d{2,4})\b']:
                    found = None
                    for m in re.finditer(pattern, html_payload, re.IGNORECASE):
                        if m.group(1) != current_num:
                            found = m
                            break
                    if found:
                        icymi_original = f'"{found.group(1)}"'
                        break

            # Construct episode_url from canonical Touch Point pattern when RSS <link> is absent.
            # Touch Point's URL slug includes the ICYMI marker, so derive it from the original
            # (unstripped) title minus only the TP### prefix.
            if not link:
                url_title = re.sub(r'^\s*TP\d+\s*[\:\-\s]*', '', title, flags=re.IGNORECASE).strip()
                link = f"https://touchpoint.health/podcast/{ep_num.lower()}-{slugify(url_title)}/"

            clean_title = title
            clean_title = re.sub(r'^\s*TP\d+\s*[\:\-\s]*', '', clean_title, flags=re.IGNORECASE)
            clean_title = re.sub(r'\bICYMI\b\s*[\:\-\s]*', '', clean_title, flags=re.IGNORECASE)
            # Strip editorial-artifact prefixes leaked from RSS (observed in TP482).
            clean_title = re.sub(r'^\s*(Episode Outline|Outline|Episode Notes)\s*:\s*', '', clean_title, flags=re.IGNORECASE)
            # Strip leading en-dash / em-dash / hyphen (observed in TP469, TP470).
            clean_title = re.sub(r'^[–—\-]\s*', '', clean_title)
            clean_title = clean_title.strip().strip('"').strip("'").strip()
            clean_title = clean_title.replace('"', '\\"')

            metadata_map[url_filename] = {
                "ep_num": ep_num,
                "title": clean_title,
                "date": pub_date_iso,
                "url": link,
                "duration": duration_hhmmss,
                "duration_seconds": duration_seconds,
                "description": indented_desc,
                "resources": resources_list,
                "is_icymi": str(is_icymi).lower(),
                "icymi_original_episode": icymi_original,
                "remote_audio_url": audio_url
            }
    except Exception as e:
        print(f"❌ Error compiling XML structural schema: {str(e)}")
    
    return metadata_map

def run_qa_audit(transcript_body, file_name):
    """Executes a dual-stage Macro/Micro heuristic audit. Returns a list of structured
    findings (each a dict with type/detail/severity/suggested_action) that callers can
    emit into the transcript's YAML frontmatter as qa_flags so downstream Phase 3 cleanup
    picks them up. Pass transcript_body (the section AFTER frontmatter, with NAME:\\ncontent
    speaker blocks) so frontmatter text doesn't pollute the 3-gram analysis."""
    findings = []
    text_clean = transcript_body.lower()

    raw_words = re.sub(r'[\*#\-\n\:]', ' ', text_clean).split()
    total_words = len(raw_words)

    # Macro-gate: 3-grams appearing >15 times across the body
    phrases_3gram = [" ".join(raw_words[i : i + 3]) for i in range(len(raw_words) - 2)]
    global_counts = {p: phrases_3gram.count(p) for p in set(phrases_3gram) if phrases_3gram.count(p) > 15}

    # Micro-audit: 3 consecutive identical 3-grams (Whisper autoregressive loop signature)
    strict_proximity_loops = set()
    for i in range(len(raw_words) - 9):
        block_a = " ".join(raw_words[i : i + 3])
        block_b = " ".join(raw_words[i + 3 : i + 6])
        block_c = " ".join(raw_words[i + 6 : i + 9])
        if block_a == block_b == block_c and len(block_a.strip()) > 0:
            strict_proximity_loops.add(block_a)

    print(f"\n📊 DUAL-STAGE DATA QUALITY AUDIT FOR: {file_name}")
    print(f"   [Total Word Volume: {total_words} words]")
    print("-" * 50)

    if global_counts:
        print("  📢 MACRO-GATE NOTICE: High frequency phrases identified:")
        for phrase, count in global_counts.items():
            print(f"    - '{phrase}' appears {count} times across total file.")

    if strict_proximity_loops:
        print("\n  🛑 MICRO-AUDIT FAILURE: Strict consecutive looping detected!")
        for loop in strict_proximity_loops:
            print(f"    - CRITICAL ERROR: Repeating loop tracking sequence: '{loop} [x3]'")
            findings.append({
                "type": "asr_loop",
                "detail": f"Repeating triplet: '{loop}' [x3]",
                "severity": "warning",
                "suggested_action": "Strip the looped span during Phase 3 cleanup",
            })
    else:
        print("  ✅ MICRO-AUDIT PASS: No consecutive autoregressive loops detected.")

    # Diarization monologue check (new NAME:\ncontent format): split body into blank-line-
    # separated blocks, check those whose first line is a NAME: label and whose remaining
    # lines exceed 450 words combined.
    long_block_speaker = None
    for block in transcript_body.split("\n\n"):
        block_lines = [ln for ln in block.split("\n") if ln.strip()]
        if len(block_lines) >= 2 and re.match(r'^[A-Z][A-Z \(\)]+:$', block_lines[0]):
            content_words = sum(len(ln.split()) for ln in block_lines[1:])
            if content_words > 450:
                long_block_speaker = block_lines[0].rstrip(":")
                break
    if long_block_speaker:
        print(f"\n  ⚠️ DIARIZATION ALERT: Massive unbroken monologue detected (>450 words) for {long_block_speaker}.")
        findings.append({
            "type": "diarization_monologue",
            "detail": f"Single turn for {long_block_speaker} exceeds 450 words",
            "severity": "info",
            "suggested_action": "Phase 3 may split into multiple paragraphs at natural break points",
        })

    # Host inversion check (works against either ALL-CAPS or mixed-case turn headers)
    for seg in transcript_body.split("\n\n"):
        seg_upper = seg.upper()
        seg_lower = seg.lower()
        if "CHRIS BOYER:" in seg_upper and any(x in seg_lower for x in ["thanks chris", "hey chris", "right chris"]):
            findings.append({
                "type": "attribution_inversion",
                "detail": "A CHRIS BOYER turn contains vocative addressing of Chris — possible speaker swap",
                "severity": "warning",
                "suggested_action": "Verify against audio; relabel turn if PyAnnote swapped speakers",
            })
            print("\n  ⚠️ ATTRIBUTION ALERT: CHRIS BOYER turn contains 'thanks/hey/right Chris' — possible inversion.")
            break
    for seg in transcript_body.split("\n\n"):
        seg_upper = seg.upper()
        seg_lower = seg.lower()
        if "REED SMITH:" in seg_upper and any(x in seg_lower for x in ["thanks reed", "hey reed", "right reed"]):
            findings.append({
                "type": "attribution_inversion",
                "detail": "A REED SMITH turn contains vocative addressing of Reed — possible speaker swap",
                "severity": "warning",
                "suggested_action": "Verify against audio; relabel turn if PyAnnote swapped speakers",
            })
            print("\n  ⚠️ ATTRIBUTION ALERT: REED SMITH turn contains 'thanks/hey/right Reed' — possible inversion.")
            break

    print("-" * 50)
    if findings:
        print(f"🛑 FINAL PIPELINE STATUS: {len(findings)} finding(s) flagged for Phase 3 cleanup. Written to qa_flags in frontmatter.")
    else:
        print("🛡️ FINAL PIPELINE STATUS: PASSED SYSTEM HEALTH CHECK")
    print("=" * 50 + "\n")

    return findings

def flush_batch(batch_files, batch_number):
    """Zip the current batch's transcripts + audio into the staging folder, then clear source files."""
    if not batch_files:
        return
    zip_name = f"touchpoint_batch_{batch_number:03d}.zip"
    zip_path = os.path.join(OUTPUT_DIR, zip_name)
    file_list = " ".join(f'"{f}"' for f in batch_files)
    transcript_count = sum(1 for f in batch_files if f.endswith(".md"))
    print("\n" + "="*70)
    print(f"📦 BATCH {batch_number}: Compressing {transcript_count} transcripts + audio into {zip_name}...")
    print("="*70)
    os.system(f'zip -q "{zip_path}" {file_list}')
    print(f"   ✅ Batch {batch_number} staged at: {zip_path}")
    print(f"      → Download via Colab sidebar: Files panel → batches → right-click {zip_name} → Download")
    # Clear local source files (transcripts + audio) — they're now inside the zip
    for f in batch_files:
        try:
            os.remove(f)
        except OSError:
            pass

# =====================================================================
# MAIN AUTOMATED PROCESSING PIPELINE EXECUTION
# =====================================================================
rss_catalog = parse_rss_metadata(XML_FILE_PATH)

if rss_catalog:
    print("\n" + "="*60)
    print("🧠 INTERACTIVE BATCH FILTER INTERFACE")
    print("="*60)
    print("Supported formats:")
    print("  - Single Episode: 489 or TP489")
    print("  - Multiple Episodes (Comma Separated): 489, 488, 487")
    print("  - Look-Back Date Barrier: 2026-05-10")
    print("-" * 60)
    
    user_query = input("👉 Enter Episode Code(s) OR a look-back date (YYYY-MM-DD): ").strip()
    
    queue = []
    if re.match(r'^\d{4}-\d{2}-\d{2}$', user_query):
        print(f"Filtering items broadcasted on or after barrier target date: {user_query}")
        for filename, meta in rss_catalog.items():
            if meta["date"] != "null" and meta["date"] >= user_query:
                queue.append((filename, meta))
    else:
        requested_episodes = []
        for ep in user_query.split(","):
            clean_ep = ep.strip().upper()
            if clean_ep.isdigit():
                clean_ep = f"TP{clean_ep}"
            requested_episodes.append(clean_ep)

        for filename, meta in rss_catalog.items():
            if meta["ep_num"] in requested_episodes:
                queue.append((filename, meta))

    if not queue:
        print("🛑 No matching episodes discovered for your selection query. Check inputs and rerun.")
    else:
        print(f"✨ Target verification complete. Queued {len(queue)} episodes for processing.\n")

        # Stage batch zips on the Colab runtime disk for manual sidebar download.
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print(f"📁 Batch zips will land in: {OUTPUT_DIR}")
        print(f"   Download each from the Colab sidebar (Files panel → batches → right-click → Download)")
        print(f"   or grab them all at once at the end of the run.\n")

        batch_files = []
        batch_number = 1

        for index, (target_file, meta) in enumerate(queue, 1):
            print("\n" + "="*70)
            print(f"📦 BATCH RUN {index}/{len(queue)}: {meta['ep_num']} — {meta['title']}")
            print("="*70)

            print(f"📥 Cloud Streaming remote audio asset directly to runtime storage...")
            try:
                urllib.request.urlretrieve(meta['remote_audio_url'], target_file)
                print("   ✅ Audio streaming download completed successfully.")
            except Exception as e:
                print(f"   ❌ Network streaming error on asset: {str(e)}. Skipping to next item.")
                continue

            print("-> Running Pipeline Step 1: Audio Transcription Engine...")
            model = whisperx.load_model("large-v3-turbo", "cuda", compute_type="float16", asr_options=ASR_OPTIONS, vad_options=VAD_OPTIONS)
            audio = whisperx.load_audio(target_file)
            result = model.transcribe(audio, batch_size=16)

            print("-> Running Pipeline Step 2: High-Resolution Timestamp Alignment...")
            model_a, metadata = whisperx.load_align_model(language_code=result["language"], device="cuda")
            result = whisperx.align(result["segments"], model_a, metadata, audio, device="cuda", return_char_alignments=False)

            del model; del model_a; gc.collect(); torch.cuda.empty_cache()

            print("-> Running Pipeline Step 3: PyAnnote Multi-Speaker Cluster Analysis...")
            diarize_model = DiarizationPipeline(token=HF_TOKEN, device="cuda")
            diarize_segments = diarize_model(audio, min_speakers=2, max_speakers=3)

            print("-> Running Pipeline Step 4: Compiling Attributions and Front-Matter Data...")
            result = whisperx.assign_word_speakers(diarize_segments, result)
            
            # FIXED: Bypassed non-fatal identification exceptions natively
            speaker_map = resolve_speaker_identities(result["segments"])

            # Per-turn Announcer leakage defense. The cluster-level collision detector
            # (resolve_speaker_identities) handles the case where most of an Announcer
            # cluster is actually a host/guest. But the residual TP475-style failure is:
            # the cluster is *labeled* Announcer (because boilerplate matched) yet
            # individual long turns inside it are substantive host/guest speech. Catch
            # those at the turn level and re-attribute to the nearest named speaker.
            ANNOUNCER_LEAK_BOILERPLATE_RE = re.compile(
                r"\bwelcome to touchpoint\b"
                r"|\ba podcast dedicated to discussions on digital marketing\b"
                r"|\bbrought to you by\b"
                r"|\bsponsored by\b"
                r"|\bthanks for listening\b"
                r"|\brate.*review\b",
                re.IGNORECASE,
            )
            turn_overrides = {}
            named_positions = [
                (i, speaker_map.get(s.get("speaker", ""), s.get("speaker", "")))
                for i, s in enumerate(result["segments"])
                if speaker_map.get(s.get("speaker", ""), s.get("speaker", "")) not in ("Announcer", "", "UNKNOWN_SPEAKER")
                and not speaker_map.get(s.get("speaker", ""), s.get("speaker", "")).startswith("SPEAKER_")
            ]
            if named_positions:
                for i, seg in enumerate(result["segments"]):
                    raw = seg.get("speaker", "UNKNOWN_SPEAKER")
                    name = speaker_map.get(raw, raw)
                    if name != "Announcer":
                        continue
                    text = (seg.get("text") or "").strip()
                    if len(text.split()) <= 25:
                        continue
                    if ANNOUNCER_LEAK_BOILERPLATE_RE.search(text):
                        continue
                    nearest_idx, nearest_name = min(named_positions, key=lambda p: abs(p[0] - i))
                    turn_overrides[i] = nearest_name
                if turn_overrides:
                    print(
                        f"  ℹ️ Per-turn Announcer leakage defense: re-attributed "
                        f"{len(turn_overrides)} long Announcer turns to nearest named speaker."
                    )

            resources_yaml_lines = []
            if meta['resources']:
                resources_yaml_lines.append("resources:")
                for res in meta['resources']:
                    safe_raw = res['raw_text'].replace('\\', '\\\\').replace('"', '\\"')
                    resources_yaml_lines.append(f"  - url: {res['url']}")
                    resources_yaml_lines.append(f'    raw_text: "{safe_raw}"')
            else:
                resources_yaml_lines.append("resources: []")
            resources_block_str = "\n".join(resources_yaml_lines)

            output = [
                "---",
                "source: Touchpoint Podcast",
                f"episode_number: {meta['ep_num']}",
                f"episode_title: \"{meta['title']}\"",
                f"episode_url: {meta['url']}",
                f"pub_date: {meta['date']}",
                f"duration: {meta['duration']}",
                f"duration_seconds: {meta['duration_seconds']}",
                "description: |",
                f"{meta['description']}",
                f"{resources_block_str}",
                f"is_icymi: {meta['is_icymi']}",
                f"icymi_original_episode: {meta['icymi_original_episode']}",
                "---",
                f"\n# Transcript: {meta['title']} ({meta['ep_num']})\n"
            ]

            transcript_body = []
            current_speaker = None
            for idx, segment in enumerate(result["segments"]):
                raw_speaker = segment.get("speaker", "UNKNOWN_SPEAKER")
                real_name = turn_overrides.get(idx) or speaker_map.get(raw_speaker, raw_speaker)
                text = segment['text'].strip()
                
                if real_name != current_speaker:
                    transcript_body.append(f"\n\n{real_name.upper()}:\n{text}")
                    current_speaker = real_name
                else:
                    transcript_body.append(f" {text}")

            export_name = f"{meta['ep_num'].lower()}_transcript.md"

            # Run QA audit on the transcript body BEFORE assembling frontmatter, so the
            # findings can be injected as qa_flags YAML. Downstream Phase 3 (clean-transcripts)
            # reads qa_flags and applies targeted cleanups (strip ASR loops, etc.).
            transcript_body_str = "".join(transcript_body)
            qa_findings = run_qa_audit(transcript_body_str, export_name)

            if qa_findings:
                qa_yaml = ["qa_flags:"]
                for f in qa_findings:
                    detail = f['detail'].replace('\\', '\\\\').replace('"', '\\"')
                    action = f['suggested_action'].replace('\\', '\\\\').replace('"', '\\"')
                    qa_yaml.append(f"  - type: {f['type']}")
                    qa_yaml.append(f"    detail: \"{detail}\"")
                    qa_yaml.append(f"    severity: {f['severity']}")
                    qa_yaml.append(f"    suggested_action: \"{action}\"")
                # Insert right before the closing '---' of the frontmatter
                output.insert(-2, "\n".join(qa_yaml))

            final_text = "\n".join(output[:-1]) + "\n" + output[-1] + transcript_body_str

            with open(export_name, "w") as f:
                f.write(final_text)
            print(f"💾 Asset written to workspace disk: {export_name}")

            # Rename the source mp3 so its basename matches the transcript (e.g. tp489_transcript.mp3)
            audio_export_name = f"{meta['ep_num'].lower()}_transcript.mp3"
            try:
                if target_file != audio_export_name:
                    if os.path.exists(audio_export_name):
                        os.remove(audio_export_name)
                    os.rename(target_file, audio_export_name)
                print(f"🔊 Audio renamed for archive: {audio_export_name}")
            except OSError as e:
                print(f"   ⚠️ Could not rename audio file ({e}); keeping original name '{target_file}'.")
                audio_export_name = target_file

            gc.collect()
            torch.cuda.empty_cache()

            batch_files.extend([export_name, audio_export_name])

            if index % BATCH_SIZE == 0:
                flush_batch(batch_files, batch_number)
                batch_files = []
                batch_number += 1

        # Flush any remaining transcripts that didn't fill a full batch
        if batch_files:
            flush_batch(batch_files, batch_number)

        total_batches = batch_number if batch_files else batch_number - 1
        print("\n" + "="*70)
        print(f"🎉 ALL PROCESSING QUEUES COMPLETE. {len(queue)} episodes processed across {total_batches} batch zip(s).")
        print(f"   All zips staged in: {OUTPUT_DIR}")
        print(f"   Right-click each in the Colab Files sidebar to download,")
        print(f"   or run this in a new cell to pull them all at once:")
        print(f"     from google.colab import files")
        print(f"     for f in sorted(os.listdir('{OUTPUT_DIR}')): files.download(os.path.join('{OUTPUT_DIR}', f))")
        print("="*70)